# 2.g) Soil suitability, global — 5 arc-min, 0.25° and EC-Earth3 grid

The soil suitability rating of `2.c)` (HWSD v2.01 × GAEZ v5 Module IV, one SSR 0–100 per soil mapping unit, by soil
component then weighted by `SHARE`; rainfed) is put on the HWSD2 raster (30 arc-sec) and **averaged** over each cell
of three grids, global:
- **5 arc-min** (the MIRCA2000 grid, 4320 × 2160);
- **0.25°** (1440 × 720);
- **EC-Earth3** (256 × 512, Gaussian latitudes, the grid of the SVM): each 30″ pixel goes to the cell that contains
  its centre (cell edges, as the MIRCA2000 pixels in 0.1).

**Mean of the SSR** over the land pixels of the cell, weighted by the pixel area (cos latitude): SSR is a continuous
rating, averaged as in `2.e)` (no threshold, no majority). Ocean (raster no-data) is left out. **Permafrost (gelic
soil units, `ADD_PROP = 2`)**: GAEZ v5 Module IV, "unsuitable for crop growing and excluded from evaluation": no SSR
in `2.c)`; here they count as **0** in the mean (unsuitable), not left out, which would give a cell half in
permafrost the rating of its other half. Their share of the cell is saved (`gelic_frac`).

Climate and land use are not in this layer: it is combined with the climate suitability later (step 5 / 6).
Crops as in the SVM: Wheat, Wheat-winter, Wheat-spring (the same soil rating: GAEZ has no edaphic difference), Maize,
Potato, Rice (wetland rice table), Soybean, rainfed.

Outputs in `<out_root>/2_soil_suitability/` (BDAP: `elisa/Crop_suitability/outputs/2_soil_suitability/`):
`soil_suitability_rf_<level>_<grid>.nc`, one variable per crop (`Maize_rf`, ...), plus `land_frac_hwsd` (share of
the cell with HWSD2 land pixels) and `gelic_frac`; maps.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb

# ---- parameters
LEVELS = ["int"]                   # GAEZ input level of the SSR: "int" (intermediate, production) | "low" | "high"
CROPS = {"Wheat": ["Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"], "Maize": ["Maize_rf"], "Potato": ["Potato_rf"],
         "Rice": ["Rice_rf"], "Soybean": ["Soybean_rf"]}   # soil crop of 2.c -> crop names of the SVM
ROWS_PER_BLOCK = 600               # 30" rows read at once (multiple of 10 and 30: whole 5' and 0.25 deg cells)
HWSD_TIF = cu.data_file(CFG, "hwsd_raster")
OUT = cu.out_dir(CFG, "2_soil_suitability")
SMU_PKL = OUT / "_soil_suitability_by_smu.pkl"   # written by 2.c
assert SMU_PKL.exists(), f"run 2.c first: {SMU_PKL}"

ssr = pd.read_pickle(SMU_PKL)
ssr = ssr[ssr.regime == "rf"]
conn = cu.hwsd_connect(cu.data_file(CFG, "hwsd_mdb"))
smu = pd.read_sql("SELECT HWSD2_SMU_ID, ADD_PROP FROM HWSD2_SMU", conn)
GELIC = smu.HWSD2_SMU_ID[smu.ADD_PROP == 2].astype(int).values
print(f"SSR of {ssr.SMU_ID.nunique()} soil units, {len(GELIC)} gelic units (permafrost, rated 0) | outputs {OUT}")

## 1. Look-up tables (soil unit → SSR) and the three grids

In [ ]:
with rasterio.open(HWSD_TIF) as src:
    W, H, NODATA, T = src.width, src.height, src.nodata, src.transform
assert (W, H) == (43200, 21600) and abs(T.a - 1 / 120) < 1e-9 and abs(T.c + 180) < 1e-6 and abs(T.f - 90) < 1e-6, "HWSD2 raster: 30 arc-sec global expected"
MAXID = int(max(ssr.SMU_ID.max(), GELIC.max(), smu.HWSD2_SMU_ID.max())) + 1
LUT = {}
for lv in LEVELS:
    for soil_crop in CROPS:
        lut = np.full(MAXID + 1, np.nan, np.float32)
        s = ssr[ssr.crop == soil_crop]
        lut[s.SMU_ID.astype(int).values] = s[f"SSR_{lv}"].astype(np.float32).values
        lut[GELIC] = 0.0                                  # permafrost: unsuitable
        LUT[(lv, soil_crop)] = lut
IS_GELIC = np.zeros(MAXID + 1, bool); IS_GELIC[GELIC] = True

lat_px = 90 - (np.arange(H) + 0.5) / 120                 # 30" pixel centres (north -> south)
lon_px = -180 + (np.arange(W) + 0.5) / 120
GRID = cu.svm_grid(CFG)
_, J_EC = cu.cell_of(np.zeros(W), lon_px, GRID)          # EC-Earth3 column of each 30" column
I_EC, _ = cu.cell_of(lat_px, np.zeros(H), GRID)          # EC-Earth3 row of each 30" row
NE = GRID["lat"].size * GRID["lon"].size
REG = {"5arcmin": 10, "0.25deg": 30}                     # regular grids: block of 30" pixels per cell
print({k: f"{H // f} x {W // f}" for k, f in REG.items()}, "| EC-Earth3", GRID["lat"].size, "x", GRID["lon"].size)

## 2. Area-weighted means, block of rows by block of rows

In [ ]:
# sums of w*SSR and of w over the land pixels (w = cos lat), per grid, crop and level; land and gelic weights
keys = list(LUT)
NAMES = ["land", "gelic", "all"] + keys + [(k, "w") for k in keys]
acc = {g: {k: np.zeros((H // f, W // f), np.float32) for k in NAMES} for g, f in REG.items()}
acc["ecearth3"] = {k: np.zeros(NE) for k in NAMES}


def add(name, a, r0, flat):
    # one layer of the block (w x value) summed into every grid, then dropped
    for g, f in REG.items():
        n = ROWS_PER_BLOCK // f
        acc[g][name][r0 // f:r0 // f + n] = a.reshape(n, f, W // f, f).sum(axis=(1, 3), dtype=np.float64)
    acc["ecearth3"][name] += np.bincount(flat, weights=a.ravel(), minlength=NE)


t0 = time.time()
with rasterio.open(HWSD_TIF) as src:
    for r0 in range(0, H, ROWS_PER_BLOCK):
        ids = src.read(1, window=Window(0, r0, W, ROWS_PER_BLOCK)).astype(np.int64)
        land = (ids != NODATA) & (ids <= MAXID)
        idc = np.where(land, ids, 0)
        w = np.repeat(np.cos(np.radians(lat_px[r0:r0 + ROWS_PER_BLOCK])).astype(np.float32)[:, None], W, axis=1)
        flat = (I_EC[r0:r0 + ROWS_PER_BLOCK, None] * GRID["lon"].size + J_EC[None, :]).ravel()
        add("all", w, r0, flat)
        add("land", land * w, r0, flat)
        add("gelic", (land & IS_GELIC[idc]) * w, r0, flat)
        for k in keys:
            v = LUT[k][idc]
            ok = land & np.isfinite(v)
            add(k, np.where(ok, v * w, np.float32(0)), r0, flat)
            add((k, "w"), ok * w, r0, flat)
        if (r0 // ROWS_PER_BLOCK) % 6 == 0:
            print(f"  rows {r0}-{r0 + ROWS_PER_BLOCK} ({90 - r0 / 120:.0f} N) ... {(time.time() - t0) / 60:.1f} min", flush=True)
print(f"done in {(time.time() - t0) / 60:.1f} min")

## 3. Save

In [ ]:
def coords_of(g):
    if g == "ecearth3":
        return GRID["lat"], GRID["lon"]
    f = REG[g]; res = f / 120
    return -90 + res * (np.arange(H // f) + 0.5), -180 + res * (np.arange(W // f) + 0.5)   # lat ascending


def field(g, a):
    a = a.reshape(GRID["lat"].size, GRID["lon"].size) if g == "ecearth3" else a[::-1]   # regular: north -> south flipped
    return a


FILES = {}
for g in acc:
    lat, lon = coords_of(g)
    A = acc[g]
    with np.errstate(invalid="ignore", divide="ignore"):
        land_frac = field(g, A["land"] / A["all"])
        gelic_frac = field(g, np.where(A["land"] > 0, A["gelic"] / A["land"], np.nan))
    for lv in LEVELS:
        dv = {}
        for soil_crop, names in CROPS.items():
            k = (lv, soil_crop)
            with np.errstate(invalid="ignore", divide="ignore"):
                m = field(g, np.where(A[(k, "w")] > 0, A[k] / A[(k, "w")], np.nan)).astype(np.float32)
            for n in names:
                dv[n] = (("lat", "lon"), m, {"long_name": f"GAEZ soil suitability rating SSR (0-100), {n}, input level {lv}",
                                             "units": "1", "soil_crop_2c": soil_crop})
        dv["land_frac_hwsd"] = (("lat", "lon"), land_frac.astype(np.float32), {"long_name": "share of the cell with HWSD2 land pixels"})
        dv["gelic_frac"] = (("lat", "lon"), gelic_frac.astype(np.float32), {"long_name": "share of the land of the cell in gelic soil units (permafrost, SSR 0)"})
        ds = xr.Dataset(dv, coords={"lat": lat, "lon": lon},
                        attrs={"source": "HWSD v2.01 x GAEZ v5 Module IV (2.c), rainfed, area-weighted mean of the 30 arc-sec pixels",
                               "grid": g, "input_level": lv, "gelic": "permafrost soil units counted as SSR 0"})
        FILES[(g, lv)] = OUT / f"soil_suitability_rf_{lv}_{g}.nc"
        ds.to_netcdf(FILES[(g, lv)])
        print("saved", FILES[(g, lv)].name, dict(ds.sizes))

## 4. Checks: the three grids give the same mean, and maps

In [ ]:
rows = []
for (g, lv), f in FILES.items():
    with xr.open_dataset(f) as ds:
        wlat = np.cos(np.radians(ds.lat)) * ds.land_frac_hwsd.fillna(0)
        for n in sum(CROPS.values(), []):
            rows.append({"grid": g, "level": lv, "crop": n,
                         "global land mean SSR": float((ds[n] * wlat).sum() / wlat.where(ds[n].notnull()).sum()),
                         "share of land with SSR >= 50 (%)": float(100 * (wlat * (ds[n] >= 50)).sum() / wlat.where(ds[n].notnull()).sum())})
check = pd.DataFrame(rows).pivot_table(index=["level", "crop"], columns="grid", values="global land mean SSR")
display(check.round(2))   # the same up to the grid: averaging is consistent between the three grids

for lv in LEVELS:
    fig, axes = plt.subplots(len(CROPS), 3, figsize=(18, 3.2 * len(CROPS)))
    for r, (soil_crop, names) in enumerate(CROPS.items()):
        for c, g in enumerate(["5arcmin", "0.25deg", "ecearth3"]):
            with xr.open_dataset(FILES[(g, lv)]) as ds:
                ds[names[0]].plot(ax=axes[r, c], vmin=0, vmax=100, cmap="YlOrBr", add_colorbar=(c == 2))
            axes[r, c].set_title(f"{names[0]} SSR ({lv}), {g}", fontsize=9); axes[r, c].set_xlabel(""); axes[r, c].set_ylabel("")
    plt.tight_layout(); plt.savefig(OUT / f"soil_suitability_rf_{lv}_maps.png", dpi=70); plt.show()